   ---
## 0. Config — all tunable knobs in one place

In [ ]:
# ── PATHS (edit to match your Colab/Drive) ──────────────────────────
VERA_PATH          = 'VERAARAB_dataset.xlsx'
ARAFACTS_PATH      = 'AraFacts.csv'
SAHEEH_EARLY_PATH  = 'saheeh_temporal_early.csv'    # from the temporal EDA notebook
SAHEEH_LATE_PATH   = 'saheeh_temporal_late.csv'     # HELD-OUT TEST — never trained on
MANUAL_CLAIMS_PATH = 'manual_stress_test_claims_50.csv'
BUCKET_B_PATH      = 'Bucket_B_Truth_CLEAN_FINAL.csv'   # NEW all-real MSA corpus

# ── DATA-MIXING KNOBS ───────────────────────────────────────────────
SEED                 = 42
VERA_CAP             = 7000   # cap VERA so it stops being ~88% of training
AF_FAKE_KEEP         = 2000   # AraFacts fake to KEEP (v2 kept only 480 — too few)
AF_FAKE_REAL_RATIO   = None   # set e.g. 3.0 to cap fake at ratio*real instead of AF_FAKE_KEEP
NEW_REAL_SUPERVISED  = 1000   # new-corpus real articles entering SUPERVISED training
EGY_FRACTION         = 0.60   # share of that budget from Egyptian sources (topical gap)
MAX_PER_SOURCE       = 80     # per-source cap (kills RT domination, ~46% of the corpus)
FP_STRESS_N          = 200    # real-only held-out slice for the false-positive stress test

# ── SHORTCUT-PATTERN FIX (expanded to the verbs that dominated v2 errors) ──
SHORTCUT_PATTERNS    = ['انتشر','تداول','هل ','أعلنت','زعم','يزعم','زاعم','يدّعي','ادعاء','مزاعم']
CONTENT_STRIP_FRAC   = 0.5    # fraction of trigger-bearing FAKE train rows to also add prefix-stripped

# ── MODEL / TRAINING ────────────────────────────────────────────────
MODEL_NAME   = 'UBC-NLP/MARBERTv2'
MAX_LENGTH   = 256            # was 128 in v2 — the truncation that fed the framing shortcut
USE_TAPT     = True           # continued MLM pretraining before fine-tuning
TAPT_DIR     = './marbertv2-tapt'
TAPT_EPOCHS  = 1              # bump to 2-3 if you have time
TAPT_BLOCK   = 256
FT_EPOCHS    = 3

# ── SOURCE → REGION MAP (for source-balancing the new corpus) ───────
EGYPTIAN_KEYS = ['Youm7','Dostor','AlMasry','Sada']
PANARAB_KEYS  = ['RT_','AlJazeera','CNN','BBC','France24']
print('✅ Config loaded')

✅ Config loaded


---
## 1. Setup

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy tqdm openpyxl datasketch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 5.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd, numpy as np, torch, random, os, re
import torch.nn as nn
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForMaskedLM, Trainer, TrainingArguments,
                          EarlyStoppingCallback, DataCollatorWithPadding,
                          DataCollatorForLanguageModeling)
from datasets import Dataset
import matplotlib.pyplot as plt

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('⚠️  Runtime > Change runtime type > T4 GPU before continuing (TAPT is slow on CPU)')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('✅ Setup complete')

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Setup complete


---
## 2. Load all datasets (incl. the new all-real MSA corpus)

In [ ]:
vera         = pd.read_excel(VERA_PATH)
arafacts     = pd.read_csv(ARAFACTS_PATH, low_memory=False)
saheeh_early = pd.read_csv(SAHEEH_EARLY_PATH, low_memory=False)
saheeh_late  = pd.read_csv(SAHEEH_LATE_PATH,  low_memory=False)
manual       = pd.read_csv(MANUAL_CLAIMS_PATH)
bucket_b     = pd.read_csv(BUCKET_B_PATH)

vera['text']         = vera['text_arabic'].astype(str)
vera['binary_label'] = vera['label'].astype(int)
vera = vera.drop_duplicates(subset=['text']).reset_index(drop=True)

for d in (saheeh_early, saheeh_late):
    assert {'text','binary_label'}.issubset(d.columns), 'Saheeh splits need text + binary_label'
manual['text'] = manual['text'].astype(str)

print(f'VERA (dedup):         {len(vera)}')
print(f'AraFacts (raw):       {len(arafacts)}')
print(f'Saheeh EARLY (train): {len(saheeh_early)}')
print(f'Saheeh LATE  (TEST):  {len(saheeh_late)}   <- held out')
print(f'Manual claims:        {len(manual)}')
print(f'Bucket_B real MSA:    {len(bucket_b)}')

VERA (dedup):         17638
AraFacts (raw):       6222
Saheeh EARLY (train): 1557
Saheeh LATE  (TEST):  668   <- held out
Manual claims:        50
Bucket_B real MSA:    1384


---
## 3. Clean & prepare the new MSA corpus
- `text` (supervised) = cleaned **summary** -> fallback title
- `text_tapt` (MLM)   = cleaned **full_text** -> fallback summary
- `region` = egyptian / pan_arab (for source-balancing); all rows are **real (0)**.

In [ ]:
def region_of(src):
    s = str(src)
    if any(k in s for k in EGYPTIAN_KEYS): return 'egyptian'
    if any(k in s for k in PANARAB_KEYS):  return 'pan_arab'
    return 'pan_arab'

def clean_basic(t):
    if not isinstance(t, str): return ''
    t = re.sub(r'http\S+', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def clean_fulltext(t):
    # content reliably follows the LAST 'واتساب' in Youm7/AlMasry exports
    if not isinstance(t, str): return ''
    if 'واتساب' in t:
        tail = t.rsplit('واتساب', 1)[-1]
        if len(tail) > 120: t = tail
    return clean_basic(t)

bucket_b['region']    = bucket_b['source'].apply(region_of)
bucket_b['text']      = bucket_b['summary'].apply(clean_basic)
bucket_b.loc[bucket_b['text'].str.len() < 10, 'text'] = bucket_b['title'].apply(clean_basic)
bucket_b['text_tapt'] = bucket_b['full_text'].apply(clean_fulltext)
bucket_b.loc[bucket_b['text_tapt'].str.len() < 40, 'text_tapt'] = bucket_b['text']
bucket_b['binary_label'] = 0

bucket_b = bucket_b[bucket_b['text'].str.len() >= 10].drop_duplicates(subset=['text']).reset_index(drop=True)
print('After cleaning/dedup:', len(bucket_b))
print('\nRegion split:'); print(bucket_b['region'].value_counts())
print('\nTop sources:');  print(bucket_b['source'].value_counts().head(8))

After cleaning/dedup: 1374

Region split:
region
pan_arab    918
egyptian    456
Name: count, dtype: int64

Top sources:
source
RT_Arabic         218
RT_Arabic_Main    208
RT_Business       135
Youm7_Politics    105
Dostor_News        99
AlJazeera_Main     85
RT_MiddleEast      80
CNN_Arabic         76
Name: count, dtype: int64


---
## 4. Leakage guard — near-dup check vs the test sets
Corpus is 2026 (after the LATE window) so it is temporally clean, but we still verify no
near-duplicate text overlaps Saheeh LATE or the manual claims; any match is dropped.

In [ ]:
from datasketch import MinHash, MinHashLSH
def shingles(text, k=4):
    toks = str(text).split()
    return {' '.join(toks[i:i+k]) for i in range(max(1, len(toks)-k+1))}
def make_mh(text, num_perm=64):
    m = MinHash(num_perm=num_perm)
    for sh in shingles(text): m.update(sh.encode('utf8'))
    return m

lsh = MinHashLSH(threshold=0.8, num_perm=64)
protected = pd.concat([saheeh_late[['text']], manual[['text']]]).reset_index(drop=True)
for i, t in enumerate(protected['text']): lsh.insert(f'test_{i}', make_mh(t))

leaks = [idx for idx, t in zip(bucket_b.index, bucket_b['text']) if lsh.query(make_mh(t))]
print(f'Near-duplicate overlaps with test sets: {len(leaks)}')
if leaks:
    bucket_b = bucket_b.drop(index=leaks).reset_index(drop=True)
    print(f'Dropped -> corpus now {len(bucket_b)}')
else:
    print('✅ No leakage detected.')

Near-duplicate overlaps with test sets: 0
✅ No leakage detected.


---
## 5. Hold out a real-only false-positive stress slice
`FP_STRESS_N` source-balanced real articles, removed from the pool and reserved. Post-dating all
training data, any flagged *fake* is direct evidence of topic-novelty bias.

In [ ]:
def balanced_take(df, n, max_per_source, seed):
    parts = [g.sample(min(len(g), max_per_source), random_state=seed) for _, g in df.groupby('source')]
    pool = pd.concat(parts) if parts else df.iloc[0:0]
    return pool.sample(min(n, len(pool)), random_state=seed)

fp_stress = balanced_take(bucket_b, FP_STRESS_N, MAX_PER_SOURCE, SEED)
bucket_b_pool = bucket_b.drop(index=fp_stress.index).reset_index(drop=True)
fp_stress = fp_stress.reset_index(drop=True)
print(f'FP stress slice: {len(fp_stress)} | remaining pool: {len(bucket_b_pool)}')

FP stress slice: 200 | remaining pool: 1174


---
## 6. Source-balancing + RT-capping (supervised real budget)
Allocate `NEW_REAL_SUPERVISED` real articles, `EGY_FRACTION` from Egyptian sources, per-source
cap so RT cannot dominate and become an "RT-register = real" shortcut.

In [ ]:
def allocate_real(df, total, egy_frac, max_per_source, seed):
    egy = df[df['region']=='egyptian']; pan = df[df['region']=='pan_arab']
    n_egy = int(round(total*egy_frac)); n_pan = total-n_egy
    out = pd.concat([balanced_take(egy, n_egy, max_per_source, seed),
                     balanced_take(pan, n_pan, max_per_source, seed)])
    if len(out) > total: out = out.sample(total, random_state=seed)
    return out.reset_index(drop=True)

new_real = allocate_real(bucket_b_pool, NEW_REAL_SUPERVISED, EGY_FRACTION, MAX_PER_SOURCE, SEED)
print(f'New supervised real: {len(new_real)}')
print(new_real['region'].value_counts())
print('\nPer-source (capped):'); print(new_real['source'].value_counts())

New supervised real: 769
region
pan_arab    400
egyptian    369
Name: count, dtype: int64

Per-source (capped):
source
Youm7_Politics     80
Dostor_News        80
RT_Business        65
Youm7_Breaking     63
RT_Arabic          61
RT_Arabic_Main     59
AlJazeera_Main     52
AlMasry_Egypt      51
BBC_Arabic         47
RT_MiddleEast      45
CNN_Arabic         42
Sada_ElBalad       31
Youm7_Economy      30
AlMasry_Latest     18
AlJazeera          17
Youm7_Culture      16
France24_Arabic    12
Name: count, dtype: int64


---
## 7. AraFacts-restore (stop downsampling the fake class)
Keep a large MSA fake pool; balance via added real + class weights. Ambiguous labels
(sarcastic / partially-fake / opinion) are dropped, not forced into a class.

In [ ]:
REAL_LABELS = {'correct','صحيح',' صحيح ','حقيقة'}
AMBIGUOUS_LABELS = {'صحيح جزئياً','opinion','مشكوك فيه','sarcastic','ساخر',' ساخر ',
                    'partially-fake','انتقائي',' انتقائي ','مجتزأ'}
def map_af(raw):
    if pd.isna(raw): return np.nan
    raw = str(raw).strip()
    if raw in REAL_LABELS:      return 0
    if raw in AMBIGUOUS_LABELS: return 2
    return 1

arafacts['text'] = arafacts['claim'].astype(str)
arafacts = arafacts.drop_duplicates(subset=['text']).reset_index(drop=True)
arafacts['lab'] = arafacts['source_label'].apply(map_af)
af = arafacts[arafacts['lab'].isin([0,1])].copy(); af['binary_label'] = af['lab'].astype(int)

af_real = af[af['binary_label']==0]; af_fake = af[af['binary_label']==1]
keep_fake = int(AF_FAKE_REAL_RATIO*len(af_real)) if AF_FAKE_REAL_RATIO else AF_FAKE_KEEP
keep_fake = min(keep_fake, len(af_fake))
af_fake_keep = af_fake.sample(keep_fake, random_state=SEED)
arafacts_mix = pd.concat([af_real, af_fake_keep])[['text','binary_label']].reset_index(drop=True)
print(f'AraFacts kept -> real={len(af_real)}, fake={len(af_fake_keep)} (v2 was 160/480)')

AraFacts kept -> real=160, fake=2000 (v2 was 160/480)


---
## 8. VERA cap + dedup (stratified, preserves VERA's label ratio)

In [ ]:
if len(vera) > VERA_CAP:
    vera_capped, _ = train_test_split(vera, train_size=VERA_CAP,
                                      stratify=vera['binary_label'], random_state=SEED)
else:
    vera_capped = vera
vera_capped = vera_capped[['text','binary_label']].reset_index(drop=True)
print(f'VERA: {len(vera)} -> {len(vera_capped)}'); print(vera_capped['binary_label'].value_counts())

VERA: 17638 -> 7000
binary_label
1    3835
0    3165
Name: count, dtype: int64


---
## 9. Assemble combined supervised set + composition report

In [ ]:
parts = {
    'VERA (dialect/social)'   : vera_capped,
    'AraFacts (MSA factcheck)': arafacts_mix,
    'Saheeh EARLY (target)'   : saheeh_early[['text','binary_label']],
    'New MSA real (formal)'   : new_real[['text','binary_label']],
}
train_df = pd.concat(parts.values()).sample(frac=1, random_state=SEED).reset_index(drop=True)

rows = [[n, len(d), int(d['binary_label'].sum()), int((d['binary_label']==0).sum())]
        for n, d in parts.items()]
comp = pd.DataFrame(rows, columns=['Source','Total','Fake','Real'])
comp.loc[len(comp)] = ['TOTAL', len(train_df), int(train_df['binary_label'].sum()),
                       int((train_df['binary_label']==0).sum())]
formal_msa = len(arafacts_mix)+len(new_real)+len(saheeh_early)
print('=== TRAINING COMPOSITION (v3) ==='); print(comp.to_string(index=False))
print(f"\nFake %:           {train_df['binary_label'].mean()*100:.1f}%")
print(f"Formal-MSA share: {formal_msa/len(train_df)*100:.1f}%  (v2 ~3%)")
print(f"VERA share:       {len(vera_capped)/len(train_df)*100:.1f}%  (v2 ~88%)")

=== TRAINING COMPOSITION (v3) ===
                  Source  Total  Fake  Real
   VERA (dialect/social)   7000  3835  3165
AraFacts (MSA factcheck)   2160  2000   160
   Saheeh EARLY (target)   1557  1135   422
   New MSA real (formal)    769     0   769
                   TOTAL  11486  6970  4516

Fake %:           60.7%
Formal-MSA share: 39.1%  (v2 ~3%)
VERA share:       60.9%  (v2 ~88%)


---
## 10. Train/val split — BEFORE any augmentation
Split first; all augmentation below touches the **training split only** (fixes v2's val leakage).

In [ ]:
train_final, val_final = train_test_split(
    train_df, test_size=0.08, stratify=train_df['binary_label'], random_state=SEED)
train_final = train_final.reset_index(drop=True); val_final = val_final.reset_index(drop=True)
print(f'Train: {len(train_final)} | Val: {len(val_final)}')

Train: 10567 | Val: 919


---
## 11. Shortcut-pattern rebalancing + content-stripping (TRAIN ONLY)
(1) oversample minority label within each trigger subset; (2) for a fraction of *fake* rows opening
with a trigger, add a copy with the prefix removed so the model must read the claim.

In [ ]:
def pattern_rate(df, p):
    has = df['text'].str.contains(p, na=False); n = int(has.sum())
    return (df.loc[has,'binary_label'].mean(), n) if n>=5 else (None, n)

print('=== Trigger bias in TRAIN before fix (50% = neutral) ===')
for p in SHORTCUT_PATTERNS:
    r,n = pattern_rate(train_final, p)
    if r is not None:
        flag = '🚨' if abs(r-.5)*2>.3 else ('⚠️' if abs(r-.5)*2>.15 else '✅')
        print(f'  "{p}": {r*100:5.1f}% fake / {n:4d} {flag}')

def rebalance(df, p, seed=SEED):
    has = df['text'].str.contains(p, na=False); sub = df[has]
    if len(sub) < 10: return df, 0
    rate = sub['binary_label'].mean()
    if abs(rate-0.5) < 0.10: return df, 0
    minlab = 0 if rate>0.5 else 1
    minor = sub[sub['binary_label']==minlab]; major = sub[sub['binary_label']!=minlab]
    need = len(major)-len(minor)
    if need>0 and len(minor)>0:
        return pd.concat([df, minor.sample(need, replace=True, random_state=seed)]), need
    return df, 0

train_aug = train_final.copy(); added = 0
for p in SHORTCUT_PATTERNS:
    train_aug, k = rebalance(train_aug, p); added += k

strip_rows = []
for p in SHORTCUT_PATTERNS:
    fh = train_final[(train_final['binary_label']==1) & (train_final['text'].str.contains(p, na=False))]
    take = fh.sample(int(len(fh)*CONTENT_STRIP_FRAC), random_state=SEED) if len(fh) else fh
    for _, row in take.iterrows():
        stripped = re.sub(r'^\s*\S*\s*'+re.escape(p), '', row['text'], count=1).strip()
        if len(stripped) > 15: strip_rows.append({'text': stripped, 'binary_label': 1})
if strip_rows:
    train_aug = pd.concat([train_aug, pd.DataFrame(strip_rows)], ignore_index=True)

train_aug = train_aug.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'\nTrain {len(train_final)} -> {len(train_aug)}  (+{added} rebalanced, +{len(strip_rows)} stripped)')
print('\n=== After fix ===')
for p in SHORTCUT_PATTERNS:
    r,n = pattern_rate(train_aug, p)
    if r is not None: print(f'  "{p}": {r*100:5.1f}% fake / {n:4d}')
train_final = train_aug

=== Trigger bias in TRAIN before fix (50% = neutral) ===
  "انتشر":   5.5% fake /  127 🚨
  "تداول":  51.8% fake /  166 ✅
  "هل ":  63.4% fake /  202 ⚠️
  "أعلنت":  31.3% fake /   67 🚨
  "زعم":  26.5% fake /   34 🚨
  "يزعم":  37.5% fake /    8 ⚠️
  "زاعم":  28.6% fake /    7 🚨
  "ادعاء":  58.3% fake /   12 ⚠️
  "مزاعم":  28.6% fake /    7 🚨

Train 10567 -> 10907  (+208 rebalanced, +132 stripped)

=== After fix ===
  "انتشر":  49.4% fake /  245
  "تداول":  57.4% fake /  197
  "هل ":  59.0% fake /  312
  "أعلنت":  54.9% fake /  102
  "زعم":  54.5% fake /   55
  "يزعم":  61.5% fake /   13
  "زاعم":  44.4% fake /    9
  "يدّعي": 100.0% fake /    7
  "ادعاء":  66.7% fake /   15
  "مزاعم":  44.4% fake /    9


---
## 12. TAPT — continued MLM pretraining on in-domain text
Unsupervised masked-LM continuation on long-form in-domain text. **Leakage-safe**: no labels, and
Saheeh LATE / manual / FP-stress excluded. Highest-leverage move for the topical/register gap.
Set `USE_TAPT = False` to skip and fine-tune from base MARBERTv2.

In [ ]:
if USE_TAPT:
    tapt_texts = (vera_capped['text'].tolist()
                  + arafacts_mix['text'].tolist()
                  + saheeh_early['text'].astype(str).tolist()
                  + bucket_b_pool['text_tapt'].tolist())     # FP slice already removed
    tapt_texts = [t for t in tapt_texts if isinstance(t,str) and len(t) > 20]
    print(f'TAPT corpus: {len(tapt_texts)} documents')

    raw = Dataset.from_dict({'text': tapt_texts})
    tok = raw.map(lambda b: tokenizer(b['text'], truncation=True, max_length=TAPT_BLOCK),
                  batched=True, remove_columns=['text'])
    def group(ex):
        concat = {k: sum(ex[k], []) for k in ex}
        n = (len(concat['input_ids'])//TAPT_BLOCK)*TAPT_BLOCK
        return {k: [v[i:i+TAPT_BLOCK] for i in range(0,n,TAPT_BLOCK)] for k,v in concat.items()}
    lm = tok.map(group, batched=True)
    print(f'TAPT blocks of {TAPT_BLOCK}: {len(lm)}')

    set_seed()
    mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device)
    mlm_args  = TrainingArguments(output_dir='./tapt_ckpt', num_train_epochs=TAPT_EPOCHS,
        per_device_train_batch_size=16, learning_rate=5e-5, weight_decay=0.01,
        warmup_ratio=0.05, logging_steps=100, save_strategy='no',
        fp16=torch.cuda.is_available(), report_to='none', seed=SEED)
    Trainer(model=mlm_model, args=mlm_args, train_dataset=lm,
            data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True,
                                                          mlm_probability=0.15)).train()
    mlm_model.save_pretrained(TAPT_DIR); tokenizer.save_pretrained(TAPT_DIR)
    print(f'✅ TAPT checkpoint -> {TAPT_DIR}')
    FT_INIT = TAPT_DIR
else:
    FT_INIT = MODEL_NAME; print('TAPT skipped')

TAPT corpus: 11777 documents


Map:   0%|          | 0/11777 [00:00<?, ? examples/s]

Map:   0%|          | 0/11777 [00:00<?, ? examples/s]

TAPT blocks of 256: 1855


pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
100,3.020757


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ TAPT checkpoint -> ./marbertv2-tapt


---
## 13. Tokenize for classification (MAX_LENGTH = 256)

In [ ]:
def to_ds(df):
    ds = Dataset.from_pandas(df[['text','binary_label']].rename(columns={'binary_label':'label'}),
                             preserve_index=False)
    return ds.map(lambda b: tokenizer(b['text'], truncation=True, max_length=MAX_LENGTH,
                                      padding=False), batched=True, remove_columns=['text'])
train_ds  = to_ds(train_final)
val_ds    = to_ds(val_final)
test_ds   = to_ds(saheeh_late[['text','binary_label']])
manual_ds = to_ds(manual[['text','binary_label']])
fp_ds     = to_ds(fp_stress[['text','binary_label']])
collator  = DataCollatorWithPadding(tokenizer=tokenizer)
def compute_metrics(p):
    logits, labels = p; preds = np.argmax(logits, axis=1)
    return {'macro_f1': f1_score(labels, preds, average='macro'), 'accuracy': (preds==labels).mean()}
print('✅ Tokenized')

Map:   0%|          | 0/10907 [00:00<?, ? examples/s]

Map:   0%|          | 0/919 [00:00<?, ? examples/s]

Map:   0%|          | 0/668 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

✅ Tokenized


---
## 14. Class weights + WeightedTrainer

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *a, **k):
        super().__init__(*a, **k); self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels'); out = model(**inputs); logits = out.logits
        w = torch.tensor(self.class_weights, dtype=torch.float, device=logits.device) \
            if self.class_weights is not None else None
        loss = nn.CrossEntropyLoss(weight=w)(logits.view(-1,2), labels.view(-1))
        return (loss, out) if return_outputs else loss

class_weights = compute_class_weight('balanced', classes=np.array([0,1]),
                                     y=train_final['binary_label'].values)
print('Class weights:', class_weights)

Class weights: [1.29567593 0.81419827]


---
## 15. Fine-tune (from the TAPT checkpoint if enabled)

In [ ]:
set_seed()
model = AutoModelForSequenceClassification.from_pretrained(FT_INIT, num_labels=2).to(device)
args = TrainingArguments(output_dir='./marbert_v3', num_train_epochs=FT_EPOCHS,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
    logging_steps=50, fp16=torch.cuda.is_available(), report_to='none', seed=SEED)
trainer = WeightedTrainer(class_weights=class_weights, model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds, data_collator=collator,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train(); print('✅ Fine-tuning complete')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ./marbertv2-tapt
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_r

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.391368,0.422303,0.798511,0.799782
2,0.277900,0.390547,0.844973,0.847661
3,0.162260,0.512094,0.869676,0.873776


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Fine-tuning complete


---
## 16. Evaluate on Saheeh Masr LATE (true zero-shot test)

In [ ]:
def softmax_fake(logits):
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return (e / e.sum(axis=1, keepdims=True))[:,1]
late_logits = trainer.predict(test_ds).predictions
late_labels = saheeh_late['binary_label'].values
late_pred05 = late_logits.argmax(1)
print('='*56); print('  Saheeh LATE — default threshold 0.5'); print('='*56)
print(f'Macro F1: {f1_score(late_labels, late_pred05, average="macro"):.4f}')
print(f'Accuracy: {(late_pred05==late_labels).mean():.4f}\n')
print(classification_report(late_labels, late_pred05, target_names=['Real','Fake']))

  Saheeh LATE — default threshold 0.5
Macro F1: 0.8812
Accuracy: 0.8817

              precision    recall  f1-score   support

        Real       0.87      0.87      0.87       311
        Fake       0.89      0.89      0.89       357

    accuracy                           0.88       668
   macro avg       0.88      0.88      0.88       668
weighted avg       0.88      0.88      0.88       668



---
## 17. Threshold tuning on validation
Pick the threshold maximizing **validation** macro-F1, then apply to LATE. Tuning on val and
reporting on the untouched test is leakage-free.

In [ ]:
val_fake = softmax_fake(trainer.predict(val_ds).predictions)
val_y = val_final['binary_label'].values
grid = np.linspace(0.30, 0.70, 41)
best_t = max(grid, key=lambda t: f1_score(val_y, (val_fake>=t).astype(int), average='macro'))
print(f'Best threshold on val: {best_t:.3f}')
late_fake = softmax_fake(late_logits); late_predT = (late_fake>=best_t).astype(int)
print('\n=== Saheeh LATE — tuned threshold ===')
print(f'Macro F1: {f1_score(late_labels, late_predT, average="macro"):.4f}')
print(f'Accuracy: {(late_predT==late_labels).mean():.4f}')
print(classification_report(late_labels, late_predT, target_names=['Real','Fake']))

Best threshold on val: 0.340

=== Saheeh LATE — tuned threshold ===
Macro F1: 0.8811
Accuracy: 0.8817
              precision    recall  f1-score   support

        Real       0.88      0.87      0.87       311
        Fake       0.89      0.89      0.89       357

    accuracy                           0.88       668
   macro avg       0.88      0.88      0.88       668
weighted avg       0.88      0.88      0.88       668



---
## 18. Real-only false-positive stress test
All `fp_stress` rows are real and post-date training. The false-positive rate measures
topic-novelty bias (mundane 2026 real news flagged fake).

In [ ]:
fp_fake = softmax_fake(trainer.predict(fp_ds).predictions)
for tag, thr in [('0.5', 0.5), (f'{best_t:.3f}', best_t)]:
    pred = (fp_fake>=thr).astype(int)
    print(f'threshold {tag}: false-positive rate = {pred.mean()*100:.1f}%  ({int(pred.sum())}/{len(pred)} flagged fake)')
print('\nLower is better — high rate = distrust of novel real topics rather than content reasoning.')

threshold 0.5: false-positive rate = 3.0%  (6/200 flagged fake)
threshold 0.340: false-positive rate = 3.0%  (6/200 flagged fake)

Lower is better — high rate = distrust of novel real topics rather than content reasoning.


---
## 19. Combined Evaluation (Saheeh LATE + 200 Real-News Articles)
**Analysis only — kept separate from the primary benchmark.** This concatenates the balanced
Saheeh LATE test (real + fake) with the 200 real-only FP-stress articles. Because the FP set is
100% real by construction, the merged set is distribution-shifted (real-majority), so its
Accuracy/F1 are **not** comparable to the Saheeh LATE benchmark and must not replace it. Reported
here purely to observe combined behaviour; Saheeh LATE (Section 16/17) remains the headline metric
and the FP rate (Section 18) remains the separate robustness metric.

In [ ]:
from sklearn.metrics import confusion_matrix

# Build the combined set from the two ALREADY-EVALUATED sources (no retraining, no leakage)
combined_df = pd.concat([saheeh_late[['text','binary_label']],
                         fp_stress[['text','binary_label']]], ignore_index=True)
combined_ds = to_ds(combined_df)
comb_logits = trainer.predict(combined_ds).predictions
comb_y    = combined_df['binary_label'].values
comb_pred = (softmax_fake(comb_logits) >= best_t).astype(int)   # tuned threshold, same as LATE

combined_macro_f1 = f1_score(comb_y, comb_pred, average='macro')
combined_acc      = (comb_pred == comb_y).mean()

print('='*66)
print('  Combined Evaluation (Saheeh LATE + 200 Real-News Articles)')
print('  ANALYSIS ONLY — distribution-shifted (real-majority); NOT the primary benchmark')
print('='*66)
print(f'Samples: {len(combined_df)}  (real={int((comb_y==0).sum())}, fake={int((comb_y==1).sum())})')
print(f'Decision threshold: {best_t:.3f}')
print(f'Accuracy: {combined_acc:.4f} | Macro F1: {combined_macro_f1:.4f}\n')
print(classification_report(comb_y, comb_pred, target_names=['Real','Fake'], zero_division=0))
print('Confusion matrix [rows = true (Real, Fake), cols = pred (Real, Fake)]:')
print(confusion_matrix(comb_y, comb_pred))

combined_df['pred'] = comb_pred
combined_df.to_csv('v3_combined_eval_predictions.csv', index=False)
print('\n✅ Saved -> v3_combined_eval_predictions.csv')

Map:   0%|          | 0/868 [00:00<?, ? examples/s]

  Combined Evaluation (Saheeh LATE + 200 Real-News Articles)
  ANALYSIS ONLY — distribution-shifted (real-majority); NOT the primary benchmark
Samples: 868  (real=511, fake=357)
Decision threshold: 0.340
Accuracy: 0.9021 | Macro F1: 0.8993

              precision    recall  f1-score   support

        Real       0.92      0.91      0.92       511
        Fake       0.87      0.89      0.88       357

    accuracy                           0.90       868
   macro avg       0.90      0.90      0.90       868
weighted avg       0.90      0.90      0.90       868

Confusion matrix [rows = true (Real, Fake), cols = pred (Real, Fake)]:
[[464  47]
 [ 38 319]]

✅ Saved -> v3_combined_eval_predictions.csv


---
## 20. Summary & save

In [ ]:
summary = pd.DataFrame({
    'Evaluation': [
        'v2 — LATE (zero-shot)',
        'v3 — LATE (thr 0.5)',
        'v3 — LATE (tuned)  [PRIMARY]',
        'v3 — FP stress (real-only)',
        'v3 — Combined (LATE + 200 real)  [analysis only]',
    ],
    'Metric': ['Macro F1','Macro F1','Macro F1','False-positive rate','Macro F1'],
    'Value': [
        0.8711,
        f1_score(late_labels, late_pred05, average='macro'),
        f1_score(late_labels, late_predT, average='macro'),
        float((fp_fake >= best_t).mean()),
        combined_macro_f1,
    ],
})
print(summary.to_string(index=False))
print('\nPRIMARY benchmark = Saheeh LATE (tuned). FP rate and Combined are separate; '
      'Combined is analysis-only and does not replace the primary number.')

trainer.save_model('./final_v3_model'); tokenizer.save_pretrained('./final_v3_model')
summary.to_csv('v3_results_summary.csv', index=False)
print('\n✅ Saved -> ./final_v3_model + result CSVs')
#Persist in Colab:
#from google.colab import drive; drive.mount('/content/drive')
#import shutil; shutil.make_archive('final_v3_model','zip','./final_v3_model')
# shutil.copy('final_v3_model.zip','/content/drive/MyDrive/final_v3_model.zip')

                                      Evaluation              Metric    Value
                           v2 — LATE (zero-shot)            Macro F1 0.871100
                             v3 — LATE (thr 0.5)            Macro F1 0.881197
                    v3 — LATE (tuned)  [PRIMARY]            Macro F1 0.881097
                      v3 — FP stress (real-only) False-positive rate 0.030000
v3 — Combined (LATE + 200 real)  [analysis only]            Macro F1 0.899263

PRIMARY benchmark = Saheeh LATE (tuned). FP rate and Combined are separate; Combined is analysis-only and does not replace the primary number.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Saved -> ./final_v3_model + result CSVs


In [ ]:
trainer.save_model('./final_v3_model')
tokenizer.save_pretrained('./final_v3_model')
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copytree('./final_v3_model', '/content/drive/MyDrive/final_v3_model', dirs_exist_ok=True)
print('✅ saved to Drive')
import shutil
from google.colab import files
shutil.make_archive('final_v3_model', 'zip', './final_v3_model')
files.download('final_v3_model.zip')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Mounted at /content/drive
✅ saved to Drive


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. install + mount + login
!pip install -q huggingface_hub transformers torch
from google.colab import drive; drive.mount('/content/drive')
from huggingface_hub import login, create_repo, upload_folder
login()  # paste a WRITE token from https://huggingface.co/settings/tokens

# 2. load the saved model from Drive
import torch, json, os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
SRC = '/content/drive/MyDrive/final_v3_model'
tok   = AutoTokenizer.from_pretrained(SRC)
model = AutoModelForSequenceClassification.from_pretrained(SRC)

# 3. SWAP the two output neurons -> index 0 becomes Fake, index 1 becomes Real
with torch.no_grad():
    model.classifier.weight.data = model.classifier.weight.data[[1, 0]].clone()
    model.classifier.bias.data   = model.classifier.bias.data[[1, 0]].clone()
model.config.id2label = {0: "Fake", 1: "Real"}     # now truthful for the swapped head
model.config.label2id = {"Fake": 0, "Real": 1}

# 4. save the relabeled model + inference metadata
OUT = '/content/drive/MyDrive/final_v3_model_fake0'
model.save_pretrained(OUT); tok.save_pretrained(OUT)
json.dump({"threshold": 0.34, "max_length": 256, "model_version": "v3-marbertv2-tapt",
           "labels": {"0": "Fake", "1": "Real"}},
          open(os.path.join(OUT, "inference_meta.json"), "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
if os.path.isfile('README_model_card.md'):
    import shutil; shutil.copy('README_model_card.md', os.path.join(OUT, 'README.md'))

# 5. create repo + upload
REPO = "sarahhh33/arabic-fake-news-marbertv2"   # <-- edit
create_repo(REPO, private=True, exist_ok=True, repo_type="model")
upload_folder(folder_path=OUT, repo_id=REPO, repo_type="model",
              commit_message="v3 MARBERTv2 (TAPT), Fake=0 / Real=1")
print(f"✅ https://huggingface.co/{REPO}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...l_fake0/model.safetensors:   0%|          |  565kB /  651MB            

✅ https://huggingface.co/sarahhh33/arabic-fake-news-marbertv2
